In [1]:
import os
import csv
import numpy as np
import nibabel as nib
from PIL import Image

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
# Path to your BraTS Training Data (GLI)
TRAIN_INPUT_PATH = "./ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

# Path to your BraTS Validation Data
VAL_INPUT_PATH = "./ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData/ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData"

# Root Output Folder
OUTPUT_ROOT = "./Brats_2D_Scan"

# File suffixes
MRI_FILE_SUFFIXES = {
    't1':   { 'in': '-t1n.nii.gz', 'out': '-t1n' },
    't1ce': { 'in': '-t1c.nii.gz', 'out': '-t1c' },
    't2':   { 'in': '-t2w.nii.gz', 'out': '-t2w' },
    'flair':{ 'in': '-t2f.nii.gz', 'out': '-t2f' },
    'mask': { 'in': '-seg.nii.gz', 'out': '-seg' }
}


In [4]:
def normalize_to_png(img_data):
    """
    Normalizes MRI intensity (0-max) to 0-255 range for PNG.
    """
    if np.max(img_data) == 0:
        return img_data.astype(np.uint8)
    
    # Clip top 1% outliers to improve contrast
    p99 = np.percentile(img_data, 99)
    img_data = np.clip(img_data, 0, p99)
    
    img_norm = (img_data - np.min(img_data)) / (np.max(img_data) - np.min(img_data))
    img_norm = (img_norm * 255).astype(np.uint8)
    return img_norm

def map_mask_to_visible_png(mask_slice):
    """
    Maps BraTS classes to specific grayscale values for visibility.
    Mappings:
    - 0 (Background) -> 0
    - 1 (NCR)        -> 85
    - 2 (ED)         -> 170
    - 4 (ET)         -> 255 (Standard BraTS label for Enhancing Tumor)
    - 3 (ET alt)     -> 255 (Handling potential pre-processed data)
    """
    new_mask = np.zeros_like(mask_slice, dtype=np.uint8)
    
    # Direct mapping using boolean indexing
    new_mask[mask_slice == 1] = 85
    new_mask[mask_slice == 2] = 170
    new_mask[mask_slice == 4] = 255
    new_mask[mask_slice == 3] = 255 # Just in case label 3 is present
    
    return new_mask

def get_best_slice_index(mask_data=None, volume_shape=None):
    """
    Determines the best slice index.
    - If mask is provided: Returns slice with largest tumor area.
    - If mask is None: Returns the middle slice (for Validation data).
    """
    if mask_data is not None:
        # Check for tumor
        tumor_counts = np.sum(mask_data > 0, axis=(0, 1))
        max_tumor_pixels = np.max(tumor_counts)
        
        if max_tumor_pixels > 0:
            return np.argmax(tumor_counts), True # Found tumor
        else:
            return mask_data.shape[2] // 2, False # Mask exists but empty
    else:
        # Fallback for Validation (No mask)
        return volume_shape[2] // 2, False # Middle slice

def process_subset(input_path, output_subfolder, log_filename, user_limit):
    """
    Generic function to process either Training or Testing datasets.
    """
    # 1. Setup Folders
    full_output_path = os.path.join(OUTPUT_ROOT, output_subfolder)
    if not os.path.exists(full_output_path):
        os.makedirs(full_output_path)

    # 2. Get Patient List
    if not os.path.exists(input_path):
        print(f"Warning: Input path not found: {input_path}")
        return

    all_patients = sorted([f for f in os.listdir(input_path) 
                           if os.path.isdir(os.path.join(input_path, f))])
    
    # 3. Handle Limit ("all" or number)
    if str(user_limit).lower() == 'all':
        patients_to_process = all_patients
        limit_str = "ALL"
    else:
        try:
            limit_num = int(user_limit)
            patients_to_process = all_patients[:limit_num]
            limit_str = str(limit_num)
        except ValueError:
            print("Invalid number input. Processing 0 files.")
            return

    print(f"\n--- Processing {output_subfolder} ({limit_str} files) ---")
    
    csv_data = []
    csv_headers = ['Patient_ID', 'Slice_Index', 'Tumor_Found', 'Modality_Count', 'Output_Path']

    for patient_id in patients_to_process:
        patient_in_dir = os.path.join(input_path, patient_id)
        
        # --- A. DETERMINE SLICE INDEX ---
        # Try to load mask to find best slice
        mask_config = MRI_FILE_SUFFIXES['mask']
        mask_path = os.path.join(patient_in_dir, f"{patient_id}{mask_config['in']}")
        
        slice_idx = 0
        tumor_found = False
        
        # We need at least one file to know the shape if mask is missing
        # Let's check t1 for shape
        t1_path = os.path.join(patient_in_dir, f"{patient_id}{MRI_FILE_SUFFIXES['t1']['in']}")
        
        if os.path.exists(mask_path):
            # Training Case (Mask available)
            mask_img = nib.load(mask_path)
            mask_data = mask_img.get_fdata()
            slice_idx, tumor_found = get_best_slice_index(mask_data=mask_data)
        elif os.path.exists(t1_path):
            # Validation Case (Mask missing, use T1 for shape)
            t1_img = nib.load(t1_path)
            slice_idx, tumor_found = get_best_slice_index(mask_data=None, volume_shape=t1_img.shape)
        else:
            print(f"Skipping {patient_id} (No valid NIfTI files found)")
            continue

        # --- B. PREPARE OUTPUT FOLDER ---
        patient_out_dir = os.path.join(full_output_path, patient_id)
        if not os.path.exists(patient_out_dir):
            os.makedirs(patient_out_dir)

        # --- C. SAVE IMAGES ---
        saved_count = 0
        for key, config in MRI_FILE_SUFFIXES.items():
            # Input path
            input_file = os.path.join(patient_in_dir, f"{patient_id}{config['in']}")
            
            # If mask is missing (common in validation), skip it gracefully
            if key == 'mask' and not os.path.exists(input_file):
                continue

            if os.path.exists(input_file):
                img = nib.load(input_file)
                data_3d = img.get_fdata()
                
                # Extract Slice
                slice_2d = data_3d[:, :, slice_idx]
                slice_2d = np.rot90(slice_2d) # Rotate for viewing

                # Normalize / Map
                if key == 'mask':
                    # Map strict classes to 0, 85, 170, 255
                    slice_img_array = map_mask_to_visible_png(slice_2d)
                else:
                    # Normalize MRI scans to 0-255
                    slice_img_array = normalize_to_png(slice_2d)

                # Save as PNG
                output_filename = f"{patient_id}{config['out']}.png"
                save_path = os.path.join(patient_out_dir, output_filename)
                
                Image.fromarray(slice_img_array).save(save_path)
                saved_count += 1

        print(f"Processed: {patient_id} (Slice {slice_idx})")

        csv_data.append({
            'Patient_ID': patient_id,
            'Slice_Index': slice_idx,
            'Tumor_Found': tumor_found,
            'Modality_Count': saved_count,
            'Output_Path': patient_out_dir
        })

    # Save CSV
    log_path = os.path.join(full_output_path, log_filename)
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()
        writer.writerows(csv_data)
    print(f"Log saved to: {log_path}")

def main():
    print("=== BraTS 2D Slice Extractor (PNG) ===")
    
    # 1. Process Training Data
    print("\n[1/2] Configuration for TRAINING Data")
    train_count = input("How many TRAINING files to process? (Enter number or 'all'): ").strip()
    
    if train_count:
        process_subset(
            input_path=TRAIN_INPUT_PATH,
            output_subfolder="Training",
            log_filename="train_dataset_log.csv",
            user_limit=train_count
        )

    # 2. Process Validation/Testing Data
    print("\n[2/2] Configuration for TESTING/VALIDATION Data")
    val_count = input("How many VALIDATION files to process? (Enter number or 'all'): ").strip()
    
    if val_count:
        process_subset(
            input_path=VAL_INPUT_PATH,
            output_subfolder="Testing",
            log_filename="testing_dataset_log.csv",
            user_limit=val_count
        )

    print("\nAll tasks completed.")

if __name__ == "__main__":
    main()

=== BraTS 2D Slice Extractor (PNG) ===

[1/2] Configuration for TRAINING Data

--- Processing Training (ALL files) ---
Processed: BraTS-GLI-00000-000 (Slice 74)
Processed: BraTS-GLI-00002-000 (Slice 78)
Processed: BraTS-GLI-00003-000 (Slice 109)
Processed: BraTS-GLI-00005-000 (Slice 100)
Processed: BraTS-GLI-00006-000 (Slice 69)
Processed: BraTS-GLI-00008-000 (Slice 71)
Processed: BraTS-GLI-00008-001 (Slice 59)
Processed: BraTS-GLI-00009-000 (Slice 90)
Processed: BraTS-GLI-00009-001 (Slice 103)
Processed: BraTS-GLI-00011-000 (Slice 91)
Processed: BraTS-GLI-00012-000 (Slice 93)
Processed: BraTS-GLI-00014-000 (Slice 40)
Processed: BraTS-GLI-00014-001 (Slice 46)
Processed: BraTS-GLI-00016-000 (Slice 69)
Processed: BraTS-GLI-00016-001 (Slice 65)
Processed: BraTS-GLI-00017-000 (Slice 61)
Processed: BraTS-GLI-00017-001 (Slice 60)
Processed: BraTS-GLI-00018-000 (Slice 91)
Processed: BraTS-GLI-00019-000 (Slice 79)
Processed: BraTS-GLI-00020-000 (Slice 73)
Processed: BraTS-GLI-00020-001 (Slice 